In [ ]:
# ============================================================
# AI MENTAL-WELLNESS TEXT SENTIMENT ANALYZER
# Google Colab - Single Cell Project
# ============================================================

# Install required libraries
!pip -q install flask pyngrok vaderSentiment

# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------
from flask import Flask, request, render_template_string
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from pyngrok import ngrok
import threading
import time

# ------------------------------------------------------------
# SENTIMENT ANALYZER
# ------------------------------------------------------------
analyzer = SentimentIntensityAnalyzer()

# ------------------------------------------------------------
# FLASK APP
# ------------------------------------------------------------
app = Flask(__name__)

# ------------------------------------------------------------
# HTML PAGE
# ------------------------------------------------------------
HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>AI Mental-Wellness Text Analyzer</title>

    <meta name="viewport" content="width=device-width, initial-scale=1">

    <style>

        * {
            box-sizing: border-box;
            margin: 0;
            padding: 0;
            font-family: Arial, sans-serif;
        }

        body {
            background: linear-gradient(135deg, #dbeafe, #f0fdf4);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }

        .container {
            width: 100%;
            max-width: 700px;
            background: white;
            padding: 35px;
            border-radius: 20px;
            box-shadow: 0 10px 35px rgba(0,0,0,0.12);
        }

        h1 {
            text-align: center;
            color: #1e3a8a;
            margin-bottom: 10px;
        }

        .subtitle {
            text-align: center;
            color: #64748b;
            margin-bottom: 25px;
        }

        textarea {
            width: 100%;
            height: 150px;
            padding: 15px;
            border: 2px solid #cbd5e1;
            border-radius: 12px;
            resize: none;
            font-size: 16px;
            outline: none;
        }

        textarea:focus {
            border-color: #2563eb;
        }

        button {
            width: 100%;
            padding: 14px;
            margin-top: 15px;
            border: none;
            border-radius: 10px;
            background: #2563eb;
            color: white;
            font-size: 17px;
            cursor: pointer;
        }

        button:hover {
            background: #1d4ed8;
        }

        .result {
            margin-top: 25px;
            padding: 20px;
            background: #f8fafc;
            border-radius: 12px;
        }

        .result h2 {
            margin-bottom: 15px;
            color: #334155;
        }

        .positive {
            color: #16a34a;
            font-weight: bold;
        }

        .neutral {
            color: #ca8a04;
            font-weight: bold;
        }

        .negative {
            color: #dc2626;
            font-weight: bold;
        }

        .score {
            margin-top: 10px;
            color: #475569;
        }

        .message {
            margin-top: 15px;
            padding: 15px;
            background: #eff6ff;
            border-left: 5px solid #2563eb;
            border-radius: 8px;
        }

        .disclaimer {
            margin-top: 25px;
            font-size: 12px;
            color: #64748b;
            text-align: center;
        }

    </style>
</head>

<body>

<div class="container">

    <h1>🧠 AI Mental-Wellness Analyzer</h1>

    <p class="subtitle">
        Express your thoughts and analyze their general sentiment.
    </p>

    <form method="POST">

        <textarea
            name="text"
            placeholder="Type how you are feeling today..."
            required>{{ text }}</textarea>

        <button type="submit">
            Analyze Sentiment
        </button>

    </form>

    {% if result %}

    <div class="result">

        <h2>Analysis Result</h2>

        <p>
            <strong>Sentiment:</strong>

            {% if sentiment == "Positive" %}
                <span class="positive">{{ sentiment }}</span>
            {% elif sentiment == "Negative" %}
                <span class="negative">{{ sentiment }}</span>
            {% else %}
                <span class="neutral">{{ sentiment }}</span>
            {% endif %}

        </p>

        <p class="score">
            <strong>Compound Score:</strong>
            {{ score }}
        </p>

        <p class="score">
            <strong>Positive:</strong> {{ positive }}%
        </p>

        <p class="score">
            <strong>Neutral:</strong> {{ neutral }}%
        </p>

        <p class="score">
            <strong>Negative:</strong> {{ negative }}%
        </p>

        <div class="message">
            <strong>Wellness Message:</strong><br><br>
            {{ message }}
        </div>

    </div>

    {% endif %}

    <div class="disclaimer">
        This tool analyzes general language sentiment. It is not a
        medical diagnostic tool or a substitute for professional help.
    </div>

</div>

</body>
</html>
"""

# ------------------------------------------------------------
# WELLNESS MESSAGE FUNCTION
# ------------------------------------------------------------
def get_wellness_message(sentiment):

    if sentiment == "Positive":
        return (
            "Your text has a generally positive tone. "
            "Keep doing activities that help you feel comfortable "
            "and connected."
        )

    elif sentiment == "Negative":
        return (
            "Your text has a generally negative tone. "
            "If you are going through a difficult time, consider "
            "talking with someone you trust or a qualified professional."
        )

    else:
        return (
            "Your text has a generally neutral tone. "
            "Taking a short break, getting enough rest, and checking "
            "in with yourself can be helpful."
        )

# ------------------------------------------------------------
# ROUTE
# ------------------------------------------------------------
@app.route("/", methods=["GET", "POST"])
def home():

    result = False
    sentiment = ""
    score = 0
    positive = 0
    neutral = 0
    negative = 0
    message = ""
    text = ""

    if request.method == "POST":

        text = request.form.get("text", "").strip()

        if text:

            scores = analyzer.polarity_scores(text)

            compound = scores["compound"]

            # Sentiment classification
            if compound >= 0.05:
                sentiment = "Positive"

            elif compound <= -0.05:
                sentiment = "Negative"

            else:
                sentiment = "Neutral"

            score = round(compound, 3)

            positive = round(scores["pos"] * 100, 2)
            neutral = round(scores["neu"] * 100, 2)
            negative = round(scores["neg"] * 100, 2)

            message = get_wellness_message(sentiment)

            result = True

    return render_template_string(
        HTML,
        result=result,
        sentiment=sentiment,
        score=score,
        positive=positive,
        neutral=neutral,
        negative=negative,
        message=message,
        text=text
    )

# ------------------------------------------------------------
# START FLASK
# ------------------------------------------------------------
def run_app():
    app.run(host="0.0.0.0", port=5000, debug=False)

thread = threading.Thread(target=run_app)
thread.start()

time.sleep(3)

# ------------------------------------------------------------
# NGROK
# ------------------------------------------------------------
print("====================================================")
print("AI MENTAL-WELLNESS SENTIMENT ANALYZER")
print("====================================================")
print("Flask server started on port 5000")
print()

# IMPORTANT:
# If ngrok asks for authentication, create a free ngrok account
# and add your authentication token.

try:

    public_url = ngrok.connect(5000)

    print("Your website is running at:")
    print(public_url)

    print()
    print("Open the above URL in a new browser tab.")

except Exception as e:

    print("Ngrok could not start.")
    print("Error:", e)
    print()
    print("If you see an authentication error, configure your")
    print("ngrok authentication token first.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.5 MB/s eta 0:00:00
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


AI MENTAL-WELLNESS SENTIMENT ANALYZER
Flask server started on port 5000



ERROR:pyngrok.process.ngrok:t=2026-10-07T14:06:16+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-07T14:06:16+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


Ngrok could not start.
Error: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

If you see an authentication error, configure your
ngrok authentication token first.


In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(5000)")
print("Open your AI Mental-Wellness Analyzer here:")
print(url)

Open your AI Mental-Wellness Analyzer here:
https://5000-m-s-kkb-usc1c2-twvhtyuqlfdj-c.us-central1-2.prod.colab.dev
